In [1]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
import numpy as np
import pickle 

In [ ]:
class  Convo:
    def __init__(self, window_size, n_filter,n_channel):
        self.window_size = window_size
        self.n_filter = n_filter
        self.n_channel = n_channel
        self.kernel = np.random.randn(window_size, window_size, n_channel, n_filter) * np.sqrt(2 / (window_size * window_size * n_channel))
        self.biases = np.zeros((n_filter))

        self.kernel_momentumn = np.zeros_like(self.kernel)
        self.biases_momentumn = np.zeros_like(self.biases)


    def forward(self, inputs):
        ws = self.window_size 
        lower = int(np.floor(ws/2))
        upper = int(np.ceil(ws/2))
        self.inputs = inputs 
        batch_size, W, H, channel = inputs.shape

        if channel != self.n_channel:
            raise ValueError("input channel does not match kernel channel")

        self.feature_maps = np.zeros((batch_size, W - ws + 1, H - ws + 1, self.n_filter))

        for b in range(batch_size):
            for f in range(self.n_filter):  
                for w in range(lower, W - upper +1 ):
                    for h in range(lower, H - upper + 1): 
                        self.feature_maps[b,w-lower,h-lower,f] = np.sum(inputs[b,w-lower:w+upper,h-lower:h+upper,:] * self.kernel[:,:,:,f]) + self.biases[f]
        self.output = self.feature_maps


    def backward(self,dvalues):
        self.d_biases = np.sum(dvalues,axis=(0,1,2))
        self.d_kernel = np.zeros_like(self.kernel)
        self.d_inputs = np.zeros_like(self.inputs)

        ws = self.window_size 
        lower = int(np.floor(ws/2))
        upper = int(np.ceil(ws/2))
        batch_size, W, H, channel = self.inputs.shape

        for b in range(batch_size):
            for f in range(self.n_filter):  
                for w in range(lower, W - upper +1 ):
                    for h in range(lower, H - upper + 1): 
                        grad = dvalues[b,w-lower,h-lower,f]
                        self.d_kernel[:,:,:,f] += self.inputs[b,w-lower:w+upper,h-lower:h+upper,:] * grad
                        self.d_inputs[b,w-lower:w+upper,h-lower:h+upper,:] += self.kernel[:,:,:,f] * grad


class Pooling:
    def __init__(self, pool_size=2, stride = 2, mode = 'max'):
        self.pool_size = pool_size
        self.stride = stride
        self.mode = mode

    def forward(self,inputs):
        self.inputs = inputs
        ps = self.pool_size
        s = self.stride
        batch_size, W, H, n_filter = inputs.shape
        output_W = int(np.floor((W - ps)/s)) + 1
        output_H = int(np.floor((H - ps)/s)) + 1

        self.output = np.zeros((batch_size, output_W, output_H, n_filter))

        for b in range(batch_size):
            for f in range(n_filter):  
                for w in range(output_W ):
                    for h in range(output_H): 
                        w_start = w*s
                        h_start = h*s
                        patch = inputs[b, w_start : w_start + ps, h_start : h_start + ps,f]
                        if self.mode == 'max':
                            self.output[b, w, h, f] = np.max(patch) 
                        elif self.mode == 'mean':
                            self.output[b, w, h, f] = np.mean(patch) 
                        else:
                            print('invalid pooling mode')
                            raise ValueError("please choose 'max' or 'mean'")
                        

    def backward(self,dvalues):

        inputs = self.inputs
        ps = self.pool_size
        s = self.stride
        batch_size, W, H, n_filter = inputs.shape
        output_W = int(np.floor((W - ps)/s)) + 1
        output_H = int(np.floor((H - ps)/s)) + 1
        self.d_inputs = np.zeros_like(inputs)

        for b in range(batch_size):
            for f in range(n_filter):  
                for w in range(output_W ):
                    for h in range(output_H): 
                        w_start = w*s
                        h_start = h*s
                        patch = inputs[b, w_start : w_start + ps, h_start : h_start + ps,f]

                        if self.mode == 'max':
                            mask = (patch == np.max(patch))
                            self.d_inputs[b, w_start:w_start+ps, h_start:h_start+ps, f] += mask * dvalues[b, w, h, f]

                        elif self.mode == 'mean':
                            self.d_inputs[b, w_start:w_start+ps, h_start:h_start+ps, f] += float(dvalues[b, w, h, f] / (self.pool_size**2))



class conv_optimizer:
    def __init__(self,lr = 0.001, beta = 0.9):
        self.lr = lr
        self.beta = beta

    def OptimizerGD(self,layer):
        layer.kernel -= self.lr*layer.d_kernel
        layer.biases -= self.lr*layer.d_biases

    def OptimizerGD_WithMomentumn(self, layer):
        
        layer.kernel_momentumn = self.beta * layer.kernel_momentumn - self.lr*layer.d_kernel
        layer.biases_momentumn = self.beta * layer.biases_momentumn - self.lr*layer.d_biases

        layer.kernel += layer.kernel_momentumn
        layer.biases += layer.biases_momentumn



In [ ]:
class LayerDense:

    def __init__(self,n_input,n_neuron):
        self.weights = np.random.randn(n_input, n_neuron) * np.sqrt(2 / n_input)
        self.biases = np.zeros((1,n_neuron))

        self.d_weights = np.zeros_like(self.weights)
        self.d_biases = np.zeros_like(self.biases)

        self.weight_momentumn = np.zeros_like(self.weights)
        self.biases_momentumn = np.zeros_like(self.biases)

        self.wgrad_squared_sum = np.zeros_like(self.biases)
        self.bgrad_squared_sum = np.zeros_like(self.weights)

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs,self.weights) + self.biases

    def backward(self, dvalues):
        self.d_biases = np.sum(dvalues,axis=0,keepdims=True)
        self.d_weights = np.dot(self.inputs.T,dvalues)
        self.d_inputs = np.dot(dvalues,self.weights.T)


class relu:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0,inputs)

    def backward(self,dvalues):
        self.d_inputs = np.where(self.inputs <= 0, 0, dvalues)

class tanh:
    def forward(self,inputs):
        self.inputs = inputs
        self.output = (np.exp(inputs) - np.exp(-inputs)) / (np.exp(inputs) + np.exp(-inputs))

    def backward(self,dvalues):
        self.d_inputs = dvalues * (1-self.output**2)


class sigmoid:
    def forward(self,inputs):
        self.inputs = inputs
        self.output = 1/ (1+np.exp(-inputs))

    def backward(self,dvalues):
        self.d_inputs = dvalues * (self.output) * (1-self.output)


class softmax:
    def forward(self,inputs):
        self.inputs = inputs
        exp_values = np.exp(inputs - np.max(inputs,axis=1,keepdims=True))
        self.output = exp_values / np.sum(exp_values,axis = 1, keepdims= True)

    def backward(self,dvalues):
        diagonal = np.diag(self.output)[:None]
        pad = np.zeros(diagonal.shape)
        sii = np.hstack((np.diagflat(diagonal),pad))
        self.d_inputs = sii - (diagonal * self.output)
        #in progress, let skip it as it's not needed


class Loss:
    def forward(self,y_pred,y_true):
        self.batch_size = y_pred.shape[0]
        self.y_pred = y_pred
        self.y_true = y_true
        p = y_pred[y_true == 1]
        p = np.clip(p,0+1e-9,1 - 1e-9)
        self.loss =  - np.average(np.log(p))
    
    def backward(self):
        y_pred = np.clip(self.y_pred,0+1e-9,1 - 1e-9)
        self.d_loss =  - self.y_true/ (y_pred*self.batch_size)

    def backward_with_softmax(self):
        self.d_inputs = (self.y_pred - self.y_true) / self.batch_size


class optimizer:
    def __init__(self,lr = 0.001, beta = 0.9, beta2 = 0.999):
        self.lr = lr
        self.beta = beta
        self.beta2 = beta2
        self.interations = 0

    def count_interation (self):
        self.interations += 1

    def OptimizerGD(self,layer):
        layer.weights -= self.lr*layer.d_weights
        layer.biases -= self.lr*layer.d_biases

    def OptimizerGD_WithMomentumn(self, layer):
        
        layer.weight_momentumn = self.beta * layer.weight_momentumn - self.lr*layer.d_weights
        layer.biases_momentumn = self.beta * layer.biases_momentumn - self.lr*layer.d_biases
 
        layer.weights += layer.weight_momentumn
        layer.biases += layer.biases_momentumn

    def adagrad(self,layer):

        layer.wgrad_squared_sum += layer.d_weights ** 2
        layer.bgrad_squared_sum += layer.d_biases ** 2

        layer.weights -= (self.lr*layer.d_weights) / (np.sqrt(layer.wgrad_squared_sum) + 1e-10)
        layer.biases -= (self.lr*layer.d_biases) / (np.sqrt(layer.bgrad_squared_sum) + 1e-10)

    def RMSprop(self,layer):

        layer.wgrad_squared_sum = (1-self.beta) * (layer.d_weights ** 2) +  self.beta * layer.wgrad_squared_sum
        layer.bgrad_squared_sum = (1-self.beta) * (layer.d_biases ** 2) +  self.beta * layer.bgrad_squared_sum

        layer.weights -= (self.lr*layer.d_weights) / (np.sqrt(layer.wgrad_squared_sum) + 1e-10)
        layer.biases -= (self.lr*layer.d_biases) / (np.sqrt(layer.bgrad_squared_sum) + 1e-10)

    def adam(self,layer):

        layer.weight_momentumn = self.beta * layer.weight_momentumn + (1-self.beta)*layer.d_weights
        layer.biases_momentumn = self.beta * layer.biases_momentumn + (1-self.beta)*layer.d_biases

        layer.wgrad_squared_sum = self.beta2 * layer.wgrad_squared_sum +  (1-self.beta2) * (layer.d_weights ** 2)
        layer.bgrad_squared_sum = self.beta2 * layer.bgrad_squared_sum +  (1-self.beta2) * (layer.d_biases ** 2)

        t = self.iterations

        Moment_bias_correction = (1-self.beta**t)
        grad_sqursum_bias_correction = (1-self.beta2**t)

        Corrected_wgrad_squared_sum = layer.wgrad_squared_sum / grad_sqursum_bias_correction
        Corrected_bgrad_squared_sum = layer.bgrad_squared_sum / grad_sqursum_bias_correction

        Corrected_weight_momentumn = layer.weight_momentumn / Moment_bias_correction
        Corrected_biases_momentumn = layer.biases_momentumn / Moment_bias_correction

        layer.weights -= (self.lr*Corrected_weight_momentumn) / (np.sqrt(Corrected_wgrad_squared_sum) + 1e-10)
        layer.biases -= (self.lr*Corrected_biases_momentumn) / (np.sqrt(Corrected_bgrad_squared_sum) + 1e-10)



        


In [4]:
def OneHotMatrix(labels,num_classes):
    labels = np.asarray(labels, dtype=int).reshape(-1)
    y_true = np.zeros((len(labels), num_classes))
    y_true[np.arange(len(labels)), labels] = 1
    return y_true


def accuracy(y_pred, y_true):
    y_pred_labels = np.argmax(y_pred, axis=1)

    if len(y_true.shape) == 2:
        y_true_labels = np.argmax(y_true, axis=1)
    else:
        y_true_labels = y_true

    return np.mean(y_pred_labels == y_true_labels)

In [6]:
class Model:
    def __init__(self,layers):
        self.layers = layers

    def save(self,filename):
        with open (filename,'wb') as f:
            pickle.dump(self,f)

    @staticmethod
    def load(filename):
        with open(filename,'rb') as f:
            return pickle.load(f)
        
    def forward(self,X):
        output = X
        for layer in self.layers:
            layer.forward(output)
            output = layer.output
        return output

class Flatten:
    def forward(self, inputs):
        self.input_shape = inputs.shape
        self.output = inputs.reshape(inputs.shape[0], -1)

    def backward(self, dvalues):
        self.d_inputs = dvalues.reshape(self.input_shape)

data

In [7]:
x = pickle.load(open('train_dts_80.pickle','rb'))
y = pickle.load(open('train_labels_80.pickle','rb'))

x_test = pickle.load(open('test_dts_80.pickle','rb'))
y_test = pickle.load(open('test_labels_80.pickle','rb'))

X = x/255.0

In [8]:
print(X.shape)  
print(y.shape)  

(8005, 80, 80, 1)
(8005,)


In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [10]:
print(X_train.shape)
print(X_val.shape)
print(x_test.shape)

print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(6404, 80, 80, 1)
(1601, 80, 80, 1)
(2023, 80, 80, 1)
(6404,)
(1601,)
(2023,)


In [11]:
def after_conv_shape(img_width, img_height, kernel_size, n_filter):
    return (img_width - kernel_size + 1, img_height - kernel_size + 1, n_filter)

def after_pool_shape(img_width, img_height, channel, pool_size, stride ):
    return ( int(np.floor((img_width - pool_size)/stride) + 1), int(np.floor((img_height - pool_size)/stride) + 1), channel )

In [12]:
#CONFIG
epoches = 1000
batch_size = 64
channel_num = X_train.shape[3]
patience = 10

In [13]:
print(X_train.shape)

(6404, 80, 80, 1)


In [14]:
print(X_train.shape)
print(y_train.shape)

(6404, 80, 80, 1)
(6404,)


In [15]:
features_c1 = after_conv_shape(X_train.shape[1],X_train.shape[2],5,8)
features_p1 = after_pool_shape(*features_c1,2,2)

features_c2 = after_conv_shape(*features_p1[:-1],3,16)
features_p2 = after_pool_shape(*features_c2,2,2)

features_c3 = after_conv_shape(*features_p2[:-1],3,32)
features_p3 = after_pool_shape(*features_c3,2,2)

last_features_flattened = int(np.prod(features_p3))

In [16]:
print(features_p3)
print(last_features_flattened)

(8, 8, 32)
2048


In [ ]:
#CONV

conv1 =  Convo(5, 8, channel_num)
conv_activation1 = relu()
pool1 = Pooling()

conv2 =  Convo(3, 16, 8)
conv_activation2 = relu()
pool2 = Pooling()

conv3 =  Convo(3, 32, 16)
conv_activation3 = relu()
pool3 = Pooling()



#MLP

ld1 = LayerDense(last_features_flattened, 32)
activation1 = relu()

ld2 = LayerDense(32, 2)
activation2 = softmax()

#LOST and OPTIMIZE

L = Loss()

opt = optimizer(lr=0.005)

conv_opt = conv_optimizer(lr=0.005)


flatten = Flatten()

lowest_Loss = np.inf

In [19]:
#resume training
model = Model.load('best_dc_model.pkl')

conv1, conv_activation1,pool1, conv2,conv_activation2, pool2, conv3,conv_activation3, pool3, flatten, ld1, activation1,ld2, activation2 = model.layers

L = Loss()

opt = optimizer(lr=0.0005)

conv_opt = conv_optimizer(lr=0.0005)


In [21]:
num_classes = len(np.unique(y))
y_val_oh = OneHotMatrix(y_val,num_classes)
pat = 0
model = Model([conv1, conv_activation1,pool1, conv2,conv_activation2, pool2, conv3,conv_activation3, pool3, flatten, ld1, activation1,ld2, activation2])



for epoch in range(epoches):

    epoch_loss = 0
    epoch_acc = 0
    train_count = 0

    indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[indices]
    y_train_shuffled = y_train[indices]


    for b in range(0,len(X_train_shuffled),batch_size):
        X_batch = X_train_shuffled[b:b+batch_size]
        y_batch = OneHotMatrix(y_train_shuffled[b:b+batch_size],num_classes)

        conv1.forward(X_batch)
        conv_activation1.forward(conv1.feature_maps)
        pool1.forward(conv_activation1.output)

        conv2.forward(pool1.output)
        conv_activation2.forward(conv2.feature_maps)
        pool2.forward(conv_activation2.output)

        conv3.forward(pool2.output)
        conv_activation3.forward(conv3.feature_maps)
        pool3.forward(conv_activation3.output)

        flatten.forward(pool3.output)

        ld1.forward(flatten.output)
        activation1.forward(ld1.output)

        ld2.forward(activation1.output)
        activation2.forward(ld2.output)


        L.forward(activation2.output,y_batch)


        acc = accuracy(activation2.output, y_batch)


        L.backward_with_softmax()

        ld2.backward(L.d_inputs)
        
        activation1.backward(ld2.d_inputs)
        ld1.backward(activation1.d_inputs)


        flatten.backward(ld1.d_inputs)


        pool3.backward(flatten.d_inputs)
        conv_activation3.backward(pool3.d_inputs)
        conv3.backward(conv_activation3.d_inputs)

        pool2.backward(conv3.d_inputs)
        conv_activation2.backward(pool2.d_inputs)
        conv2.backward(conv_activation2.d_inputs)

        pool1.backward(conv2.d_inputs)
        conv_activation1.backward(pool1.d_inputs)
        conv1.backward(conv_activation1.d_inputs)
        

        opt.OptimizerGD_WithMomentumn(ld2)
        opt.OptimizerGD_WithMomentumn(ld1)
        conv_opt.OptimizerGD_WithMomentumn(conv3)
        conv_opt.OptimizerGD_WithMomentumn(conv2)
        conv_opt.OptimizerGD_WithMomentumn(conv1)

        epoch_loss += L.loss * len(X_batch)
        epoch_acc += acc * len(X_batch)
        train_count += len(X_batch)


    epoch_avg_loss = epoch_loss / train_count
    epoch_avg_acc = epoch_acc/train_count

    epoch_val_loss = 0
    epoch_val_acc = 0
    val_count = 0


    #val
    for b in range(0,len(X_val),batch_size):
        X_val_batch = X_val[b:b+batch_size]
        y_val_batch = y_val_oh[b:b+batch_size]

        conv1.forward(X_val_batch)
        conv_activation1.forward(conv1.feature_maps)
        pool1.forward(conv_activation1.output)

        conv2.forward(pool1.output)
        conv_activation2.forward(conv2.feature_maps)
        pool2.forward(conv_activation2.output)

        conv3.forward(pool2.output)
        conv_activation3.forward(conv3.feature_maps)
        pool3.forward(conv_activation3.output)


        flatten.forward(pool3.output)


        ld1.forward(flatten.output)
        activation1.forward(ld1.output)


        ld2.forward(activation1.output)
        activation2.forward(ld2.output)


        L.forward(activation2.output,y_val_batch)

        epoch_val_loss += L.loss * len(X_val_batch)
        epoch_val_acc += accuracy(activation2.output, y_val_batch) * len(X_val_batch)
        val_count += len(X_val_batch)

    val_loss = epoch_val_loss / val_count
    val_acc = epoch_val_acc/val_count



    if val_loss >= lowest_Loss:
        pat += 1
        if pat >= patience :
            break 
        
    else:
        pat = 0
        lowest_Loss = val_loss
        model.save('best_dc_model.pkl')
        print('best model saved')


    print(f"Epoch {epoch+1}: train_acc: {epoch_avg_acc:.4f}, train_loss = {epoch_avg_loss:.4f}")
    print(f"Epoch {epoch+1}: val_acc: {val_acc:.4f}, val_loss = {val_loss:.4f}")





best model saved
Epoch 1: train_acc: 0.8271, train_loss = 0.3906
Epoch 1: val_acc: 0.7452, val_loss = 0.5468
best model saved
Epoch 2: train_acc: 0.8323, train_loss = 0.3809
Epoch 2: val_acc: 0.7464, val_loss = 0.5360
best model saved
Epoch 3: train_acc: 0.8335, train_loss = 0.3793
Epoch 3: val_acc: 0.7552, val_loss = 0.5300
Epoch 4: train_acc: 0.8306, train_loss = 0.3808
Epoch 4: val_acc: 0.7408, val_loss = 0.5383
Epoch 5: train_acc: 0.8393, train_loss = 0.3688
Epoch 5: val_acc: 0.7414, val_loss = 0.5370
Epoch 6: train_acc: 0.8295, train_loss = 0.3785
Epoch 6: val_acc: 0.7502, val_loss = 0.5338
Epoch 7: train_acc: 0.8393, train_loss = 0.3647
Epoch 7: val_acc: 0.7495, val_loss = 0.5357
Epoch 8: train_acc: 0.8465, train_loss = 0.3561
Epoch 8: val_acc: 0.7533, val_loss = 0.5344
Epoch 9: train_acc: 0.8467, train_loss = 0.3518
Epoch 9: val_acc: 0.7445, val_loss = 0.5445
Epoch 10: train_acc: 0.8493, train_loss = 0.3525
Epoch 10: val_acc: 0.7533, val_loss = 0.5390
Epoch 11: train_acc: 0.8393

KeyboardInterrupt: 

In [17]:
def predict_in_batches(model, X, batch_size=32):
    outputs = []
    for i in range(0, len(X), batch_size):
        X_batch = X[i:i+batch_size]
        out = model.forward(X_batch)
        outputs.append(out.copy())  
    return np.vstack(outputs)

In [18]:
dc_model = Model.load('best_dc_model.pkl')
y_pred = predict_in_batches(dc_model, x_test, batch_size=32)
dc_acc = accuracy(y_pred, y_test)
print(f"test accuracy: {dc_acc*100}%")

test accuracy: 68.16608996539793%
